In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/path/to/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

# Ch4 — From document to graph

We turn `data/annual_report.md` into a trained, queryable `GMSExpertStore`. Ingestion is **deterministic** (regex mode, no LLM augmentation) so the graph is byte-stable, and every authoritative number is parsed once into Exact Numerical Memory (ENM) rather than left as a string a model might re-read. `build_rag_store` runs GEODE self-correction (Ch5) before training; here we read its diagnostics and confirm the store round-trips.

In [ ]:
import torch

from knowlytix.core.config import GeometryConfig, TrainConfig
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.geode.rag import build_rag_store
from knowlytix.knowledge.geode.loop import make_default_trainer

REPO_ROOT = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
MD = os.path.join(REPO_ROOT, "data", "annual_report.md")
STORE = os.path.join(REPO_ROOT, "data", "gms_annual_report_store")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = DocGMSConfig(
    store_path=STORE,
    ingest_mode="regex",   # deterministic: no model-extracted numbers
    loss_mode="cap",
    geometry=GeometryConfig(d_v=64, d_u=64, m=32, d=32),
    train=TrainConfig(epochs=150, batch_size=64, neg_samples=16,
                      lr=5e-3, lr_riemannian=2e-3),
)

## Listing 1 — `build_rag_store` over the annual report

One call ingests the document, runs the GEODE correction loop, trains the production GMS, populates ENM, and saves the store to disk. We print the entity / relation / triple / ENM counts and the GEODE audit (corrections, anchor violations). **GPU/Qwen cell — the lead executes this in CI; do not run it during authoring.**

In [ ]:
res = build_rag_store(
    MD, config, device=device,
    geode_trainer=make_default_trainer(device, epochs=80),
)
store = res.store

print(f"converged={res.converged}  iterations={res.iterations}")
print(f"entities={res.n_entities}  triples={res.n_triples}  enm={res.n_enm}")
print(f"relations={len(store.adapter.relation_to_idx)}")
print(f"GEODE corrections:      {len(res.corrections)}")
print(f"GEODE anchor violations:{len(res.anchor_violations)}")

Expected output on the shipped corpus:

```
converged=True  iterations=1
entities=... triples=... enm=21
relations=10
GEODE corrections:      0
GEODE anchor violations:0
```

Ten relations — `has_amount`, `has_division`, `has_fy2024`, `has_fy2025`, `has_head`, `has_headcount`, `has_region`, `has_revenue`, `has_value`, `in_section` — and **21** ENM entries (the segment, income-statement, and balance-sheet figures). Because the clean corpus has no contradictions or broken sum anchors, GEODE reports zero corrections and zero anchor violations; Ch5 corrupts a figure to make the loop work.

## Listing 2 — reload from disk and query

The store is just files under `store_path`. We construct a fresh `GMSExpertStore` from the same config and `load()` it — no rebuild, no GEODE, no training — then read a figure from ENM and pattern-match a triple. ENM is the authoritative numeric channel: `lookup_enm` returns the value **byte-exact**, never a string parsed at query time.

In [ ]:
from knowlytix.knowledge.store import GMSExpertStore

reloaded = GMSExpertStore(config, device)
assert reloaded.load(), f'no store at {STORE}'

# Exact numeric recall (segment_performance category, byte-exact).
cloud_rev = reloaded.lookup_enm("segment_performance",
                                "Cloud Platform/Technology/Revenue")
total_rev = reloaded.lookup_enm("income_statement", "Revenue/FY2025")
print(f"Cloud Platform revenue = {cloud_rev}")
print(f"Total FY2025 revenue   = {total_rev}")

# Pattern-match the relational triple that links the segment to its division.
div = reloaded.query_triples(head="cloud platform", relation="has_division")
print("cloud platform division:", div)

Expected output:

```
Cloud Platform revenue = 120.0
Total FY2025 revenue   = 355.0
cloud platform division: [('cloud platform', 'has_division', 'technology')]
```

The reloaded store answers the same as the freshly built one: the figures match the report's Segment Performance and Income Statement tables, and the `cloud platform -> technology` edge is present for the multi-hop chains in Ch8.

## Exercise — add a row, rebuild, retrieve

Add a fifth segment to the report, rebuild, and confirm the new fact is retrievable. We append `Media | Technology | 40.0 | 90` to the Segment Performance table (writing to a copy so the shipped corpus stays untouched), rebuild into a scratch store, and query the new segment's revenue. **GPU/Qwen cell — lead executes in CI.**

In [ ]:
import shutil, tempfile

with open(MD, encoding='utf-8') as f:
    text = f.read()
# Insert the new row directly above the Total row of the segment table.
new_row = "| Media | Technology | 40.0 | 90 |\n"
total_line = "| Total | All | 355.0 | 1500 |"
augmented = text.replace(total_line, new_row + total_line)
assert new_row in augmented

scratch_dir = tempfile.mkdtemp(prefix='ch4_aug_')
aug_md = os.path.join(scratch_dir, 'annual_report_aug.md')
with open(aug_md, 'w', encoding='utf-8') as f:
    f.write(augmented)

import dataclasses
aug_config = dataclasses.replace(
    config, store_path=os.path.join(scratch_dir, 'store'))
aug = build_rag_store(aug_md, aug_config, device=device,
                      geode_trainer=make_default_trainer(device, epochs=80))
media_rev = aug.store.lookup_enm('segment_performance',
                                 'Media/Technology/Revenue')
print('new segment Media revenue =', media_rev)
assert media_rev == 40.0
shutil.rmtree(scratch_dir)

## Self-check — the built store round-trips and has the segments

Proves the chapter's claim: a store built from the document can be saved and reloaded byte-for-byte (same ENM value, same segment triples). The asserted figures come straight from `corpus_facts.md`. **GPU cell — lead executes in CI.**

In [ ]:
# Round-trip: the reloaded store agrees with the freshly built one
# on the authoritative figures, and carries the four segments.
for cat, key, expected in [
    ('segment_performance', 'Cloud Platform/Technology/Revenue', 120.0),
    ('segment_performance', 'Devices/Technology/Revenue', 80.0),
    ('segment_performance', 'Logistics/Operations/Revenue', 95.0),
    ('segment_performance', 'Retail/Operations/Revenue', 60.0),
    ('segment_performance', 'Total/All/Revenue', 355.0),
]:
    assert store.lookup_enm(cat, key) == expected
    assert reloaded.lookup_enm(cat, key) == expected

segments = {h for (h, r, t) in reloaded.triples if r == 'has_revenue'}
assert {'cloud platform', 'devices', 'logistics', 'retail'} <= segments
assert reloaded.stats()['enm_entries'] == store.stats()['enm_entries'] == 21
print('OK: store round-trips; 4 segments present; 21 ENM entries')